# High-Growth Account Profile (ICP from client history)

- Question: what can we observe early in a client relationship that predicts whether an account reaches 5 seats, how fast, and what happens after?
- Reader: ICP / Marketing lead and Cloudstaff leadership
- Period: clients whose first staff started 1 Jan 2021 to 7 Oct 2026 (extract date); AMER and APAC analysed separately; OPTI clients assigned to a region
- Sources: `Marketing/SC Engagement List2026-10-07 07_48_09.xlsx` (seats), `data/raw/manager_movements/mystaff client list.xlsx` (region, industry), `data/raw/monika_incremental_seats/2026 New Monthly Billed Report.xlsx` (role titles)
- Plan: `work/icp_growth_potential/plan.md`

Run from top to bottom with Run All. Every number reported for this analysis comes from a cell below.

## 1. Setup
Imports and paths. The engagement list is read in place from `Marketing/` (creating a copy under `data/raw/` is blocked by a workspace rule); the other two files are read from `data/raw/`. Charts go to `outputs/charts/icp_growth_potential/` and exports to `outputs/csv/icp_growth_potential/`.

In [ ]:
import os, sys, subprocess, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

ROOT = Path.cwd()
while not (ROOT / "data").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)

TOPIC = "icp_growth_potential"
CHARTS = ROOT / "outputs" / "charts" / TOPIC
EXPORTS = ROOT / "outputs" / "csv" / TOPIC
CHARTS.mkdir(parents=True, exist_ok=True)
EXPORTS.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT / ".claude" / "skills" / "cs-data-analytics" / "scripts"))
sys.path.insert(0, str(ROOT / "work" / TOPIC))
import icp_prep as P
import stats_helpers as st
from profile_data import load_source

# Cloudstaff Design System chart colours and type
BLUE, TEAL, YELLOW, PINK, PURPLE = "#007AFF", "#00AB9C", "#FFC426", "#E34678", "#4121A9"
TEXT, MUTED, BORDER = "#252938", "#737C90", "#DCE3EE"
plt.rcParams.update({"font.family": "Arial", "axes.edgecolor": BORDER, "axes.labelcolor": TEXT, "xtick.color": TEXT,
                     "ytick.color": TEXT, "axes.titleweight": "bold", "axes.titlecolor": "#006DE4",
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": BORDER})
REGIONS = ["AMER", "APAC"]
GROUPS = ["1 Fast 5+", "2 Slow 5+", "3 Not reached 5, active", "4 Churned, never 5"]
for f in [P.ENG_SRC, P.MYSTAFF_SRC, P.BILLED_SRC]:
    print(f, "| exists:", (ROOT / f).exists())

## 2. Load the raw engagement list
One row per staff stint on a client shortcode, exactly as received.

In [ ]:
df, meta = load_source(P.ENG_SRC)
print(f"Rows: {len(df):,} | Columns: {df.shape[1]} | SHA-256 {meta['sha256'][:16]}")
print("Engagement dates:", df["Engagement Date"].min().date(), "to", df["Engagement Date"].max().date())
df.head()

## 3. Data issues and agreed treatments
`icp_prep.prepare` applies exactly the treatments approved in `plan.md`: internal units, the Disengaged Staff Pool, test records, blank shortcodes and the duplicate row are excluded; clients flagged Internal in MyStaff or outside APAC/AMER are excluded; clients that started before 2021 are outside the cohort. OPTI clients get a region from office country, then timezone, then billing currency. Stints that end before they start are kept for the start date but not counted as seats. It also matches role titles and maps them to function families.

In [ ]:
rows, cl, ms, roles = P.prepare(df)
cl["age"] = cl["age_days"]
issues = pd.Series({
    "Disengagement before Engagement Date": int((~rows["valid"]).sum()),
    "Engagement ID reused": int(df["Engagement ID"].duplicated(keep=False).sum()),
    "Engagement Date after extract date": int((rows["ed"] > P.EXTRACT).sum()),
    "In-scope stints with no matched role": int((rows["inc"] & rows["role"].isna()).sum()),
}, name="rows_affected")
print(issues.to_string(), "\n")
print(rows["scope"].value_counts().to_string())
print(f"\nIncluded rows: {int(rows['inc'].sum()):,} of {len(rows):,} | clients in scope: {len(cl):,}")
pd.crosstab(cl["region"], cl["brand"], margins=True)

## 4a. Outcome groups
Fast = 5 concurrent seats within 183 days of the first staff start; Slow = 5+ later; Not reached 5 = still has staff today; Churned = no staff today and never reached 5.

In [ ]:
og = pd.crosstab(cl["group"], cl["region"]).reindex(GROUPS)
og_share = og / og.sum()
prof = cl.groupby(["region", "group"]).agg(clients=("peak", "size"), avg_seats_d30=("seats_d30", "mean"),
                                            pct_start_0_1=("seats_d30", lambda s: (s <= 1).mean()), avg_peak=("peak", "mean"),
                                            avg_current=("current", "mean"), pct_10plus=("peak", lambda s: (s >= 10).mean()),
                                            avg_age_months=("age", lambda s: s.mean() / 30.4375)).round(3)
tenure3 = pd.crosstab(pd.cut(cl.loc[cl.group == GROUPS[2], "age"], [-1, 364, 729, 1e9], labels=["Under 12 months", "12-23 months", "24+ months"]),
                      cl.loc[cl.group == GROUPS[2], "region"])
medians = cl.groupby(["region", "group"])[["seats_d30", "peak", "current"]].median()
display(og.join(og_share.add_suffix(" share").round(3))); display(tenure3); display(medians)
prof

## 4b. Speed to 5
Share of clients reaching 5 within each window, counting only clients old enough to have had the full window; then the time taken by clients that did reach 5.

In [ ]:
WIN = [("Within 1 month", 30), ("Within 3 months", 91), ("Within 6 months", 183), ("Within 12 months", 365),
       ("Within 18 months", 548), ("Within 24 months", 730), ("Within 36 months", 1095)]
speed = pd.DataFrame({rg: [((cl["region"] == rg) & (cl["age"] >= d) & (cl["days_to5"] <= d)).sum() /
                           ((cl["region"] == rg) & (cl["age"] >= d)).sum() for _, d in WIN] for rg in REGIONS},
                     index=[w for w, _ in WIN]).round(4)
bands = pd.cut(cl["days_to5"], [-1, 30, 90, 183, 365, 730, 1e9],
               labels=["Within 30 days", "31-90 days", "91-183 days", "6-12 months", "12-24 months", "Over 24 months"])
time5 = pd.crosstab(bands, cl["region"], normalize="columns").round(4)
speed_test = {lab: st.two_proportion_test(int(((cl.region == "AMER") & (cl.age >= d) & (cl.days_to5 <= d)).sum()), int(((cl.region == "AMER") & (cl.age >= d)).sum()),
                                         int(((cl.region == "APAC") & (cl.age >= d) & (cl.days_to5 <= d)).sum()), int(((cl.region == "APAC") & (cl.age >= d)).sum()))["p_value"]
              for lab, d in [("Within 6 months", 183), ("Within 12 months", 365)]}
display(speed); print("AMER vs APAC p-values:", {k: round(v, 5) for k, v in speed_test.items()})
time5

## 4c. Early signals (first 90 days)
Outcome: reached 5 within 12 months, clients aged 12+ months. Seats at day 30 for everyone; the other signals only for clients still under 5 seats at day 90. Chi-square tests per region.

In [ ]:
obs12 = cl["age"] >= 365
u5 = cl["seats_d90"] < 5
cl["r5_12"] = (cl["days_to5"] <= 365).astype(int)
def rate_by(col_values, mask):
    col_values = pd.Series(np.asarray(col_values), index=cl.index)
    out = []
    for rg in REGIONS:
        d = cl[obs12 & mask & (cl["region"] == rg)]
        d = d.assign(level=col_values.loc[d.index].astype(str))
        g = d.groupby("level")["r5_12"].agg(["size", "sum"])
        g["rate"] = g["sum"] / g["size"]
        g[["low", "high"]] = [st.wilson_ci(k, n) for k, n in zip(g["sum"], g["size"])]
        p = st.chi_square(g[["sum"]].assign(no=g["size"] - g["sum"]).values.tolist())["p_value"]
        out.append(g.assign(region=rg, chi_p=p).reset_index())
    return pd.concat(out).round(4)
s30 = rate_by(pd.cut(cl["seats_d30"], [-1, 0, 1, 2, 4, 1e9], labels=["0", "1", "2", "3-4", "5+"]), cl.index == cl.index)
sig_adds = rate_by(np.where(cl["adds_31_90"] >= 1, "1+", "none"), u5)
sig_fn = rate_by(np.where(cl["functions_by_d90"] >= 2, "2+", "0-1"), u5)
sig_seats90 = rate_by(pd.cut(cl["seats_d90"], [-1, 0, 1, 2, 4], labels=["0", "1", "2", "3-4"]), u5)
sig_leave = rate_by(np.where(cl["leavers_90"] >= 1, "1+", "none"), u5)
start_year = rate_by(cl["start"].dt.year, cl.index == cl.index)
rtg_start = rate_by(np.where(cl["rtg_d30"] >= 1, "yes", "no"), cl.index == cl.index)
for name, t in [("Seats at day 30", s30), ("Seats at day 90 (under 5)", sig_seats90), ("Adds days 31-90 (under 5)", sig_adds),
                ("Functions by day 90 (under 5)", sig_fn), ("Leavers first 90 days (under 5)", sig_leave),
                ("Start year", start_year), ("RTG staff in first 30 days", rtg_start)]:
    print(name); display(t)

## 4d. Industry and initial function
Industries and functions with fewer than 30 clients (aged 12+ months) in a region are grouped, as in the workbook.

In [ ]:
ind_n = cl[obs12].groupby(["region", "industry"]).size()
cl["industry_group"] = [("Unknown industry" if i == "Unknown" else i) if ind_n.get((r, i), 0) >= 30 else "Other industries"
                        for r, i in zip(cl["region"], cl["industry"])]
fn_n = cl[obs12].groupby(["region", "initial_function"]).size()
cl["initial_function_group"] = [f if (fn_n.get((r, f), 0) >= 30 or f == "Unknown") else "Other functions (under 30 clients)"
                                for r, f in zip(cl["region"], cl["initial_function"])]
industry = rate_by(cl["industry_group"], cl.index == cl.index)
initfn = rate_by(cl["initial_function_group"], cl.index == cl.index)
display(industry.sort_values(["region", "size"], ascending=[True, False]))
initfn.sort_values(["region", "size"], ascending=[True, False])

## 4e. Expansion path
Adds after day 30 split into same function vs new function (against the client's initial function), and time to a second function.

In [ ]:
exp_path = cl.groupby(["region", "group"]).agg(clients=("peak", "size"), same=("adds_Same function", "sum"),
                                                new=("adds_New function", "sum"), unknown=("adds_Unknown", "sum"),
                                                pct_clients_new_fn=("adds_New function", lambda s: (s > 0).mean()),
                                                avg_months_to_2nd=("days_to_2nd_fn", lambda s: s.mean() / 30.4375))
exp_path["pct_known_adds_new"] = exp_path["new"] / (exp_path["same"] + exp_path["new"])
before5 = cl[cl["date5"].notna()].groupby(["region", "group"])["date2_before5"].apply(lambda s: (s == "Yes").mean())
display(exp_path.round(3))
before5.round(3)

## 4f. After reaching 5
Retention 12 and 24 months after the day a client reached 5; share still with staff today by peak size (clients aged 24+ months) with a two-proportion test of peak 5+ vs under 5; how far 5+ clients grow.

In [ ]:
ret5 = {}
for rg in REGIONS:
    d = cl[(cl["region"] == rg) & cl["date5"].notna()]
    ret5[rg] = {f"active_{m}m": (d.loc[d[c] != "Too early", c] == "Active").mean()
                for m, c in [(12, "status_5p365"), (24, "status_5p730"), (36, "status_5p1095")]}
ret5 = pd.DataFrame(ret5).round(4)
a24 = cl[cl["age"] >= 730].copy()
a24["peak_band"] = pd.cut(a24["peak"], [-1, 1, 2, 4, 9, 1e9], labels=["1", "2", "3-4", "5-9", "10+"])
peakret = a24.groupby(["region", "peak_band"]).agg(clients=("peak", "size"), still_active=("current", lambda s: (s > 0).mean())).round(4)
ptest = {rg: st.two_proportion_test(int((a24[(a24.region == rg) & (a24.peak >= 5)]["current"] > 0).sum()), int(((a24.region == rg) & (a24.peak >= 5)).sum()),
                                    int((a24[(a24.region == rg) & (a24.peak < 5)]["current"] > 0).sum()), int(((a24.region == rg) & (a24.peak < 5)).sum()))
         for rg in REGIONS}
grow5 = cl[cl["date5"].notna()].groupby(["region", "group"]).agg(clients=("peak", "size"), pct_10=("peak", lambda s: (s >= 10).mean()),
                                                                  pct_20=("peak", lambda s: (s >= 20).mean()), avg_peak=("peak", "mean"),
                                                                  pct_churned=("current", lambda s: (s == 0).mean())).round(3)
display(ret5); display(peakret)
print({rg: round(v["p_value"], 6) for rg, v in ptest.items()})
grow5

## 4f2. Path to 10+
Reached 10 concurrent seats within 24 months, clients aged 24+ months, by seats at day 90 and by whether the client reached 5 within 6 months.

In [ ]:
a24c = cl[cl["age"] >= 730].copy()
a24c["r10"] = (a24c["days_to10"] <= 730).astype(int)
a24c["d90_band"] = pd.cut(a24c["seats_d90"], [-1, 1, 2, 4, 1e9], labels=["0-1", "2", "3-4", "5+"])
path10 = a24c.groupby(["region", "d90_band"])["r10"].agg(["size", "mean"]).round(4)
path10_fast = a24c.groupby(["region", a24c["days_to5"] <= P.FAST_DAYS])["r10"].agg(["size", "mean"]).round(4)
display(path10)
path10_fast

## 4g. Predictive model
Logistic regression per region on clients under 5 seats at day 90 and aged 12+ months; outcome reached 5 within 12 months. Odds ratios with 95% ranges, and a 5-fold cross-validated AUC (seed 42), the same as the workbook.

In [ ]:
import statsmodels.api as sm
FEATS = {"Seats at day 90 (0-4)": lambda d: d["seats_d90"].astype(float),
         "Staff added days 31-90 (capped at 3)": lambda d: d["adds_31_90"].clip(upper=3).astype(float),
         "2+ functions hired by day 90": lambda d: (d["functions_by_d90"] >= 2).astype(float),
         "Any leaver in first 90 days": lambda d: (d["leavers_90"] >= 1).astype(float),
         "OPTI brand": lambda d: (d["brand"] == "OPTI").astype(float)}
def auc(y, s):
    y, s = np.asarray(y), np.asarray(s); pos, neg = s[y == 1], s[y == 0]
    return float(((pos[:, None] > neg[None, :]).sum() + 0.5 * (pos[:, None] == neg[None, :]).sum()) / (len(pos) * len(neg)))
model_rows, fit_rows = [], []
for rg in REGIONS:
    d = cl[(cl["region"] == rg) & obs12 & u5]
    Xm = pd.DataFrame({k: f(d) for k, f in FEATS.items()}, index=d.index); y = (d["days_to5"] <= 365).astype(int)
    fit = sm.Logit(y, sm.add_constant(Xm)).fit(disp=0); ci = fit.conf_int()
    fold = np.random.default_rng(42).permutation(len(d)) % 5; cv = np.zeros(len(d))
    for k in range(5):
        tr, te = fold != k, fold == k
        cv[te] = sm.Logit(y[tr], sm.add_constant(Xm[tr])).fit(disp=0).predict(sm.add_constant(Xm[te], has_constant="add"))
    for k in FEATS:
        model_rows.append({"region": rg, "signal": k, "odds_ratio": np.exp(fit.params[k]), "low": np.exp(ci.loc[k, 0]),
                           "high": np.exp(ci.loc[k, 1]), "p": fit.pvalues[k]})
    fit_rows.append({"region": rg, "clients": len(d), "reached5": int(y.sum()), "cv_auc": auc(y, cv)})
model_tbl, fit_tbl = pd.DataFrame(model_rows).round(4), pd.DataFrame(fit_rows).round(3)
display(model_tbl)
fit_tbl

## 5. Charts and exports
Cloudstaff-branded charts saved to `outputs/charts/icp_growth_potential/`; the client-level table and the main result tables saved to `outputs/csv/icp_growth_potential/`.

In [ ]:
def save(fig, name):
    fig.tight_layout(); fig.savefig(CHARTS / f"{name}.png", dpi=160); plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 4.5))
for rg, c in zip(REGIONS, [BLUE, TEAL]):
    ax.plot(speed.index, speed[rg], marker="o", color=c, lw=2.5, label=rg)
ax.set_title("Cumulative share of clients reaching 5 seats"); ax.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1))
ax.legend(frameon=False); plt.setp(ax.get_xticklabels(), rotation=20); save(fig, "01_speed_to_5")

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), sharey=True)
for ax, rg in zip(axes, REGIONS):
    t = s30[s30.region == rg]
    ax.bar(t["level"].astype(str), t["rate"], color=BLUE, yerr=[t["rate"] - t["low"], t["high"] - t["rate"]], ecolor=MUTED, capsize=3)
    ax.set_title(f"{rg}: reached 5 within 12 months"); ax.set_xlabel("Seats at day 30")
    ax.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1))
save(fig, "02_reach5_by_seats_day30")

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), sharey=True)
for ax, rg in zip(axes, REGIONS):
    t = peakret.loc[rg]
    ax.bar(t.index.astype(str), t["still_active"], color=[MUTED, MUTED, MUTED, TEAL, TEAL])
    ax.set_title(f"{rg}: still with staff today"); ax.set_xlabel("Peak seats (clients aged 24+ months)")
    ax.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1))
save(fig, "03_retention_by_peak_seats")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
pts = [("Day 30", 30), ("Day 90", 90), ("Day 180", 180), ("Month 12", 365), ("Month 24", 730), ("Month 36", 1095)]
for ax, rg in zip(axes, REGIONS):
    for g, c in zip(GROUPS, [BLUE, TEAL, YELLOW, PINK]):
        ys = [cl[(cl.region == rg) & (cl.group == g) & (cl.age >= d)][f"seats_d{d}"].mean() for _, d in pts]
        ax.plot([p for p, _ in pts], ys, marker="o", color=c, lw=2.2, label=g)
    ax.set_title(f"{rg}: average seats over time")
axes[0].legend(frameon=False, fontsize=8); save(fig, "04_seat_trajectory")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6), sharex=True)
for ax, rg in zip(axes, REGIONS):
    t = initfn[(initfn.region == rg)].sort_values("rate")
    ax.barh(t["level"], t["rate"], color=[TEAL if n >= 30 else BORDER for n in t["size"]])
    ax.set_title(f"{rg}: reached 5 in 12 months by initial function", fontsize=10)
    ax.xaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1)); ax.tick_params(axis="y", labelsize=8)
save(fig, "05_reach5_by_initial_function")

out = cl.drop(columns=["client_row_pos"]).copy()
out.insert(0, "client_name", df["Client Name"].groupby(rows["sc"]).first().reindex(out.index))
out.to_csv(EXPORTS / "clients_scored.csv", index_label="shortcode")
for name, t in [("speed_to_5", speed), ("outcome_groups", og), ("early_signals_seats_d30", s30), ("industry", industry),
                ("initial_function", initfn), ("retention_by_peak", peakret), ("model", model_tbl)]:
    t.to_csv(EXPORTS / f"{name}.csv")
print(sorted(p.name for p in CHARTS.iterdir())); print(sorted(p.name for p in EXPORTS.iterdir()))

## 6. Build the workbook and run QA
Runs `work/icp_growth_potential/build_icp_growth_potential.py`, which writes the formula-driven workbook to `outputs/excel/` and prints the QA verdict (about 3 minutes: Excel recalculates every formula).

In [ ]:
res = subprocess.run([sys.executable, f"work/{TOPIC}/build_{TOPIC}.py"],
                     env={**os.environ, "CS_QA_DIR": "qa"}, capture_output=True, text=True)
print(res.stdout[-3000:]); print(res.stderr[-3000:])
assert res.returncode == 0, "Build failed"

## 7. Headline numbers
Every figure reported to the requester or in the findings comes from this table.

In [ ]:
def r(t, rg, lvl): return float(t[(t.region == rg) & (t.level.astype(str) == lvl)]["rate"].iloc[0])
H = [("Clients in scope", len(cl))]
for rg in REGIONS:
    H += [(f"{rg} clients", int((cl.region == rg).sum())),
          (f"{rg} reached 5 within 6 months (aged 6+ months)", speed.loc["Within 6 months", rg]),
          (f"{rg} reached 5 within 12 months (aged 12+ months)", speed.loc["Within 12 months", rg]),
          (f"{rg} reached 5 within 30 days, share of those that reached 5", time5.loc["Within 30 days", rg]),
          (f"{rg} reach-5 rate, 1 seat at day 30", r(s30, rg, "1")), (f"{rg} reach-5 rate, 3-4 seats at day 30", r(s30, rg, "3-4")),
          (f"{rg} under 5 at d90: added staff d31-90", r(sig_adds, rg, "1+")), (f"{rg} under 5 at d90: no adds", r(sig_adds, rg, "none")),
          (f"{rg} under 5 at d90: 2+ functions", r(sig_fn, rg, "2+")), (f"{rg} under 5 at d90: 0-1 functions", r(sig_fn, rg, "0-1")),
          (f"{rg} odds ratio per seat at day 90", float(model_tbl[(model_tbl.region == rg) & (model_tbl.signal == "Seats at day 90 (0-4)")]["odds_ratio"].iloc[0])),
          (f"{rg} model cross-validated AUC", float(fit_tbl[fit_tbl.region == rg]["cv_auc"].iloc[0])),
          (f"{rg} industry chi-square p", float(industry[industry.region == rg]["chi_p"].iloc[0])),
          (f"{rg} initial function chi-square p", float(initfn[initfn.region == rg]["chi_p"].iloc[0])),
          (f"{rg} still active, peak 1 seat (aged 24+ months)", peakret.loc[(rg, "1"), "still_active"]),
          (f"{rg} still active, peak 3-4", peakret.loc[(rg, "3-4"), "still_active"]),
          (f"{rg} still active, peak 5-9", peakret.loc[(rg, "5-9"), "still_active"]),
          (f"{rg} still active, peak 10+", peakret.loc[(rg, "10+"), "still_active"]),
          (f"{rg} active 12 months after reaching 5", ret5.loc["active_12m", rg]),
          (f"{rg} active 24 months after reaching 5", ret5.loc["active_24m", rg]),
          (f"{rg} active 36 months after reaching 5", ret5.loc["active_36m", rg]),
          (f"{rg} reached 10 in 24m, 0-1 seats at day 90", path10.loc[(rg, "0-1"), "mean"]),
          (f"{rg} reached 10 in 24m, fast 5+ clients", path10_fast.loc[(rg, True), "mean"]),
          (f"{rg} reached 5 after 12-24 months, share of 5+", time5.loc["12-24 months", rg]),
          (f"{rg} group 3 under 12 months old", tenure3.loc["Under 12 months", rg]),
          (f"{rg} group 3 24+ months old", tenure3.loc["24+ months", rg]),
          (f"{rg} median peak seats, fast 5+", medians.loc[(rg, "1 Fast 5+"), "peak"]),
          (f"{rg} fast 5+: added a new function", exp_path.loc[(rg, "1 Fast 5+"), "pct_clients_new_fn"]),
          (f"{rg} churned never 5: added a new function", exp_path.loc[(rg, "4 Churned, never 5"), "pct_clients_new_fn"])]
H += [("Role match rate, in-scope stints", float(rows.loc[rows.inc, "role"].notna().mean())),
      ("AMER vs APAC p, reached 5 within 6 months", speed_test["Within 6 months"]),
      ("AMER vs APAC p, reached 5 within 12 months", speed_test["Within 12 months"])]
headline = pd.DataFrame(H, columns=["metric", "value"])
headline["value"] = headline["value"].astype(float).round(4)
headline.to_csv(EXPORTS / "headline_numbers.csv", index=False)
headline